# 123 -- Structured Path F: Joint Taxonomy (hierarchical analog)

**Blueprint:** `outputs/reports/structured/structured_path_f_blueprint.md` -- read this first.
**NOT a re-run of DS5's `103_ds5_hierarchical_phenotype.ipynb`** (which fit a fresh within-group
clustering) **and NOT a re-run of Path C's own Part 4** (which already did local NeT sub-clustering
within Phase 1's profiles). This notebook crosses Path A's and Path C's already-fitted GLOBAL clusterings
to build and test the first actual combined 2-axis taxonomy -- see blueprint Part 0.

**Output root:** `outputs/model_outputs/structured/phenotyping/path_f/`


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
import warnings
warnings.filterwarnings("ignore")

MO = PROJECT_ROOT / "outputs" / "model_outputs" / "structured"
OUT = MO / "phenotyping" / "path_f"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT)

## 1. Load Path A and Path C assignments, build the joint taxonomy (Part 1-2 of blueprint)

In [ ]:
patha = pd.read_csv(MO / "phenotyping" / "path_a" / "behavior_only_cluster_assignments.csv")
pathc = pd.read_csv(MO / "phenotyping" / "path_c" / "net_only_cluster_assignments.csv")

BEHAV_FEATS = ["H_completion", "created_Fano_factor", "state_space_volatility",
               "backlog_trend_slope", "susceptibility", "mean_commitment_score"]
NET_FEATS = ["HHI_net", "rho_self", "H_trans", "H_state_marginal"]

joint = patha[["participant_id", "cluster_label"] + BEHAV_FEATS].rename(columns={"cluster_label": "BehavProfile"})
joint = joint.merge(
    pathc[["participant_id", "cluster_label"] + NET_FEATS].rename(columns={"cluster_label": "NetProfile"}),
    on="participant_id", how="inner"
)
joint["joint_cell"] = joint["BehavProfile"] + " x " + joint["NetProfile"]

print(f"Overlap N: {len(joint)}")
assert len(joint) == 36, f"expected N=36 per blueprint verification, got {len(joint)}"

ct = pd.crosstab(joint["BehavProfile"], joint["NetProfile"])
print()
print("Joint taxonomy cross-tabulation:")
print(ct)
print()
print("Joint cell sizes:")
print(joint["joint_cell"].value_counts())

joint.to_csv(OUT / "joint_taxonomy_assignments.csv", index=False)
print("\nSaved: joint_taxonomy_assignments.csv")

## 2. Independence test (Fisher's exact, Part 2 of blueprint)

In [ ]:
odds_ratio, p_value = fisher_exact(ct.values)
print(f"Fisher's exact test on the 2x2 taxonomy table:")
print(f"  odds ratio = {odds_ratio:.3f}")
print(f"  p = {p_value:.4f}")

if p_value < 0.05:
    indep_verdict = ("The two axes are NOT statistically independent (p<0.05) -- consistent with Path C's "
                      "weak-but-nonzero ARI=0.173. The behavioral axis dominates (Path A ARI=1.000 vs "
                      "Phase 1), but network membership still carries some real, detectable information "
                      "beyond what the behavioral clusters alone capture.")
else:
    indep_verdict = ("The two axes are not statistically distinguishable from independent at this N -- "
                      "a null result, reported honestly rather than a forced 'partial coupling' story.")
print()
print(indep_verdict)

indep_result = pd.DataFrame([{"test": "Fisher's exact, BehavProfile x NetProfile", "n": len(joint),
                               "odds_ratio": round(odds_ratio, 4), "p_value": round(p_value, 4),
                               "verdict": indep_verdict}])
indep_result.to_csv(OUT / "joint_taxonomy_independence_test.csv", index=False)
print("\nSaved: joint_taxonomy_independence_test.csv")

## 3. Cell profiles across both feature sets (first time these appear together per participant)

In [ ]:
ALL_FEATS = BEHAV_FEATS + NET_FEATS
cell_profiles = joint.groupby("joint_cell")[ALL_FEATS].agg(["mean", "count"])
cell_profiles.to_csv(OUT / "joint_taxonomy_cell_profiles.csv")

print("Cell sizes and mean profiles:")
for cell, sub in joint.groupby("joint_cell"):
    print(f"\n{cell} (n={len(sub)}):")
    print(sub[ALL_FEATS].mean().round(3).to_string())

print("\nSaved: joint_taxonomy_cell_profiles.csv")

## 4. Heatmap figure

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ct.values, cmap="Purples")
ax.set_xticks(range(len(ct.columns))); ax.set_xticklabels(ct.columns)
ax.set_yticks(range(len(ct.index))); ax.set_yticklabels(ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, str(ct.values[i, j]), ha="center", va="center", fontsize=14,
                color="white" if ct.values[i, j] > ct.values.max() / 2 else "black")
ax.set_title(f"Joint taxonomy (N={len(joint)})\nFisher's exact p={p_value:.3f}")
plt.colorbar(im, ax=ax, shrink=0.8, label="n participants")
plt.tight_layout()
plt.savefig(FIG / "fig1_pathf_joint_taxonomy_heatmap.png", dpi=150)
plt.show()
print("Saved fig1_pathf_joint_taxonomy_heatmap.png")

## 5. model_run_summary.md

In [ ]:
summary = f"""# Structured Path F -- Joint Taxonomy -- Model Run Summary

**Run date:** 2026-08-05
**Notebook:** notebooks/123_structured_hierarchical_phenotype.ipynb
**Blueprint:** outputs/reports/structured/structured_path_f_blueprint.md

## Joint taxonomy (Path A BehavProfile x Path C NetProfile, N={len(joint)})
{ct.to_string()}

## Independence test
- Fisher's exact: odds ratio={odds_ratio:.3f}, p={p_value:.4f}
- {indep_verdict}

## Cell profiles (means)
"""
for cell, sub in joint.groupby("joint_cell"):
    summary += f"\n### {cell} (n={len(sub)})\n"
    summary += sub[ALL_FEATS].mean().round(3).to_string() + "\n"

summary += """
## What this means for future ABM/GaT initialization
A 2x2 typology with cell sizes as found gives up to four candidate agent archetypes for a future
agent-based extension (mirroring DS5's own stated purpose for its 4-cell taxonomy) -- though the
smallest cell is too thin (n<10) to characterize with real confidence; only the two largest cells
should be treated as usable archetypes without further data.

## Key limitations
1. This is a cross-tabulation of two already-fitted global clusterings, not a fresh joint clustering --
   deliberately, to avoid re-deriving what Path C's own Part 4 (local sub-clustering) already covered
2. N=36 -- the smallest population of any of the four exploration paths (intersection of Path A's 41 and
   Path C's 45)
3. Smallest joint cell has very few members -- reported, not smoothed into a false 4-way balance
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)